In [7]:
## Import the necessary tools for our work
# Updated imports for Qiskit 2.0.2
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plotter  # Added for compatibility with original code
import seaborn as sns

# Visualization settings
sns.set_style("dark")
pi = np.pi

## Code for inverse Quantum Fourier Transform
## adapted from Qiskit Textbook at qiskit.org/textbook
def qft_dagger(circ_, n_qubits):
    """n-qubit QFTdagger the first n qubits in circ"""
    for qubit in range(int(n_qubits/2)):
        circ_.swap(qubit, n_qubits-qubit-1)
    for j in range(0, n_qubits):
        for m in range(j):
            circ_.cp(-np.pi/float(2**(j-m)), m, j)
        circ_.h(j)

## Code for initial state of Quantum Phase Estimation
## adapted from Qiskit Textbook at qiskit.org/textbook
## Note that the starting state is created by applying 
## H on the first n_qubits, and setting the last qubit to |psi> = |1>
def qpe_pre(circ_, n_qubits):
    circ_.h(range(n_qubits))
    circ_.x(n_qubits)

    for x in reversed(range(n_qubits)):
        for _ in range(2**(n_qubits-1-x)):
            circ_.cp(1, n_qubits-1-x, n_qubits)

## Run a Qiskit job on simulators (updated for Qiskit 2.0.2)
def run_job(circ, backend, shots=1000, optimization_level=0):
    """
    Updated function to run quantum circuits on simulators
    Removed deprecated assemble() and qobj usage
    """
    t_circ = transpile(circ, backend, optimization_level=optimization_level)
    # Direct execution without qobj (deprecated in 2.0+)
    job = backend.run(t_circ, shots=shots)
    return job.result().get_counts()

# Initialize simulator (updated for Qiskit 2.0.2)
simulator = AerSimulator()

## Function to estimate pi
## Summary: using the notation in the Qiskit textbook (qiskit.org/textbook),
## do quantum phase estimation with the 'phase' operator U = p(theta) and |psi> = |1>
## such that p(theta)|1> = exp(2 x pi x i x theta)|1>
## By setting theta = 1 radian, we can solve for pi
## using 2^n x 1 radian = most frequently measured count = 2 x pi
def get_pi_estimate(n_qubits):
    # create the circuit
    circ = QuantumCircuit(n_qubits + 1, n_qubits)
    # create the input state
    qpe_pre(circ, n_qubits)
    # apply a barrier
    circ.barrier()
    # apply the inverse fourier transform
    qft_dagger(circ, n_qubits)
    # apply a barrier
    circ.barrier()
    # measure all but the last qubits
    circ.measure(range(n_qubits), range(n_qubits))

    # run the job and get the results
    counts = run_job(circ, backend=simulator, shots=10000, optimization_level=0)
    # print(counts) 

    # get the count that occurred most frequently
    max_counts_result = max(counts, key=counts.get)
    max_counts_result = int(max_counts_result, 2)
    
    # solve for pi from the measured counts
    theta = max_counts_result/2**n_qubits
    return (1./(2*theta))

# Main execution function
def main():
    """Main function to estimate pi using different numbers of qubits"""
    print("Quantum Phase Estimation for π")
    print("=" * 50)
    print("Using QPE with U = p(θ), |ψ⟩ = |1⟩, θ = 1 radian")
    print("Formula: π = 1/(2θ) where θ = measured_value/2^n")
    print("=" * 50)
    
    # estimate pi using different numbers of qubits
    nqs = list(range(2, 12+1))
    pi_estimates = []
    
    print(f"{'Qubits':>6} {'π Estimate':>12} {'Error':>10} {'% Error':>8}")
    print("-" * 50)
    
    for nq in nqs:
        try:
            thisnq_pi_estimate = get_pi_estimate(nq)
            pi_estimates.append(thisnq_pi_estimate)
            
            error = abs(thisnq_pi_estimate - np.pi)
            percent_error = (error / np.pi) * 100
            
            print(f"{nq:>6d} {thisnq_pi_estimate:>12.6f} {error:>10.6f} {percent_error:>7.2f}%")
            
        except Exception as e:
            print(f"{nq:>6d} Error: {str(e)}")
    
    print("-" * 50)
    print(f"Actual π = {np.pi:.10f}")
    
    if pi_estimates:
        best_estimate = min(pi_estimates, key=lambda x: abs(x - np.pi))
        best_qubit_count = nqs[pi_estimates.index(best_estimate)]
        print(f"Best estimate: {best_estimate:.6f} (using {best_qubit_count} qubits)")
        print(f"Mean estimate: {np.mean(pi_estimates):.6f}")
        print(f"Std deviation: {np.std(pi_estimates):.6f}")
    
    return pi_estimates

# Additional utility functions for analysis
def plot_results(nqs, pi_estimates):
    """Plot the pi estimates vs number of qubits using original GitHub styling"""
    plotter.figure(figsize=(10, 6))
    plotter.plot(nqs, [pi]*len(nqs), '--r')
    plotter.plot(nqs, pi_estimates, '.-', markersize=12)
    plotter.xlim([1.5, 12.5])
    plotter.ylim([1.5, 4.5])
    

def analyze_single_run(n_qubits, show_circuit=False):
    """Analyze a single QPE run in detail"""
    print(f"\nDetailed Analysis for {n_qubits} qubits:")
    print("-" * 40)
    
    # Create and show circuit if requested
    circ = QuantumCircuit(n_qubits + 1, n_qubits)
    qpe_pre(circ, n_qubits)
    circ.barrier()
    qft_dagger(circ, n_qubits)
    circ.barrier()
    circ.measure(range(n_qubits), range(n_qubits))
    
    if show_circuit:
        print("Circuit:")
        print(circ.draw())
    
    # Run and analyze results
    counts = run_job(circ, backend=simulator, shots=10000, optimization_level=0)
    
    print(f"Top 5 measurement outcomes:")
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    for i, (bitstring, count) in enumerate(sorted_counts[:5]):
        decimal = int(bitstring, 2)
        theta = decimal / (2**n_qubits)
        pi_est = 1/(2*theta) if theta > 0 else float('inf')
        print(f"  {i+1}. {bitstring} (dec: {decimal:3d}) -> θ={theta:.4f} -> π≈{pi_est:.4f} ({count:4d} shots)")
    
    # Calculate final estimate
    max_result = max(counts, key=counts.get)
    max_decimal = int(max_result, 2)
    theta = max_decimal / (2**n_qubits)
    pi_estimate = 1/(2*theta) if theta > 0 else float('inf')
    
    print(f"\nFinal estimate: π ≈ {pi_estimate:.6f}")
    print(f"Error: {abs(pi_estimate - np.pi):.6f}")
    
    return pi_estimate, counts

# Run the main analysis
if __name__ == "__main__":
    # Run the main estimation
    pi_estimates = main()
    
    # Optional: Show detailed analysis for a specific number of qubits
    print("\n" + "="*60)
    analyze_single_run(8, show_circuit=False)
    
    # Optional: Create plot with original GitHub styling
    nqs = list(range(2, 12+1))
    plot_results(nqs, pi_estimates)
, 'estimate of $\pi

def analyze_single_run(n_qubits, show_circuit=False):
    """Analyze a single QPE run in detail"""
    print(f"\nDetailed Analysis for {n_qubits} qubits:")
    print("-" * 40)
    
    # Create and show circuit if requested
    circ = QuantumCircuit(n_qubits + 1, n_qubits)
    qpe_pre(circ, n_qubits)
    circ.barrier()
    qft_dagger(circ, n_qubits)
    circ.barrier()
    circ.measure(range(n_qubits), range(n_qubits))
    
    if show_circuit:
        print("Circuit:")
        print(circ.draw())
    
    # Run and analyze results
    counts = run_job(circ, backend=simulator, shots=10000, optimization_level=0)
    
    print(f"Top 5 measurement outcomes:")
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    for i, (bitstring, count) in enumerate(sorted_counts[:5]):
        decimal = int(bitstring, 2)
        theta = decimal / (2**n_qubits)
        pi_est = 1/(2*theta) if theta > 0 else float('inf')
        print(f"  {i+1}. {bitstring} (dec: {decimal:3d}) -> θ={theta:.4f} -> π≈{pi_est:.4f} ({count:4d} shots)")
    
    # Calculate final estimate
    max_result = max(counts, key=counts.get)
    max_decimal = int(max_result, 2)
    theta = max_decimal / (2**n_qubits)
    pi_estimate = 1/(2*theta) if theta > 0 else float('inf')
    
    print(f"\nFinal estimate: π ≈ {pi_estimate:.6f}")
    print(f"Error: {abs(pi_estimate - np.pi):.6f}")
    
    return pi_estimate, counts

# Run the main analysis
if __name__ == "__main__":
    # Run the main estimation
    pi_estimates = main()
    
    # Optional: Show detailed analysis for a specific number of qubits
    print("\n" + "="*60)
    analyze_single_run(8, show_circuit=False)
    
    # Optional: Create plot (uncomment if you want to see the plot)
    # nqs = list(range(2, 12+1))
    # plot_results(nqs, pi_estimates)

    plotter.xlabel('Number of qubits', fontdict={'size':20})
    plotter.ylabel('$\pi$ and estimate of $\pi

def analyze_single_run(n_qubits, show_circuit=False):
    """Analyze a single QPE run in detail"""
    print(f"\nDetailed Analysis for {n_qubits} qubits:")
    print("-" * 40)
    
    # Create and show circuit if requested
    circ = QuantumCircuit(n_qubits + 1, n_qubits)
    qpe_pre(circ, n_qubits)
    circ.barrier()
    qft_dagger(circ, n_qubits)
    circ.barrier()
    circ.measure(range(n_qubits), range(n_qubits))
    
    if show_circuit:
        print("Circuit:")
        print(circ.draw())
    
    # Run and analyze results
    counts = run_job(circ, backend=simulator, shots=10000, optimization_level=0)
    
    print(f"Top 5 measurement outcomes:")
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    for i, (bitstring, count) in enumerate(sorted_counts[:5]):
        decimal = int(bitstring, 2)
        theta = decimal / (2**n_qubits)
        pi_est = 1/(2*theta) if theta > 0 else float('inf')
        print(f"  {i+1}. {bitstring} (dec: {decimal:3d}) -> θ={theta:.4f} -> π≈{pi_est:.4f} ({count:4d} shots)")
    
    # Calculate final estimate
    max_result = max(counts, key=counts.get)
    max_decimal = int(max_result, 2)
    theta = max_decimal / (2**n_qubits)
    pi_estimate = 1/(2*theta) if theta > 0 else float('inf')
    
    print(f"\nFinal estimate: π ≈ {pi_estimate:.6f}")
    print(f"Error: {abs(pi_estimate - np.pi):.6f}")
    
    return pi_estimate, counts

# Run the main analysis
if __name__ == "__main__":
    # Run the main estimation
    pi_estimates = main()
    
    # Optional: Show detailed analysis for a specific number of qubits
    print("\n" + "="*60)
    analyze_single_run(8, show_circuit=False)
    
    # Optional: Create plot (uncomment if you want to see the plot)
    # nqs = list(range(2, 12+1))
    # plot_results(nqs, pi_estimates)
, fontdict={'size':20})
    plotter.tick_params(axis='x', labelsize=12)
    plotter.tick_params(axis='y', labelsize=12)
    plotter.show()

def analyze_single_run(n_qubits, show_circuit=False):
    """Analyze a single QPE run in detail"""
    print(f"\nDetailed Analysis for {n_qubits} qubits:")
    print("-" * 40)
    
    # Create and show circuit if requested
    circ = QuantumCircuit(n_qubits + 1, n_qubits)
    qpe_pre(circ, n_qubits)
    circ.barrier()
    qft_dagger(circ, n_qubits)
    circ.barrier()
    circ.measure(range(n_qubits), range(n_qubits))
    
    if show_circuit:
        print("Circuit:")
        print(circ.draw())
    
    # Run and analyze results
    counts = run_job(circ, backend=simulator, shots=10000, optimization_level=0)
    
    print(f"Top 5 measurement outcomes:")
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    for i, (bitstring, count) in enumerate(sorted_counts[:5]):
        decimal = int(bitstring, 2)
        theta = decimal / (2**n_qubits)
        pi_est = 1/(2*theta) if theta > 0 else float('inf')
        print(f"  {i+1}. {bitstring} (dec: {decimal:3d}) -> θ={theta:.4f} -> π≈{pi_est:.4f} ({count:4d} shots)")
    
    # Calculate final estimate
    max_result = max(counts, key=counts.get)
    max_decimal = int(max_result, 2)
    theta = max_decimal / (2**n_qubits)
    pi_estimate = 1/(2*theta) if theta > 0 else float('inf')
    
    print(f"\nFinal estimate: π ≈ {pi_estimate:.6f}")
    print(f"Error: {abs(pi_estimate - np.pi):.6f}")
    
    return pi_estimate, counts

# Run the main analysis
if __name__ == "__main__":
    # Run the main estimation
    pi_estimates = main()
    
    # Optional: Show detailed analysis for a specific number of qubits
    print("\n" + "="*60)
    analyze_single_run(8, show_circuit=False)
    
    # Optional: Create plot (uncomment if you want to see the plot)
    # nqs = list(range(2, 12+1))
    # plot_results(nqs, pi_estimates)

SyntaxError: unterminated string literal (detected at line 241) (3540346605.py, line 241)